In [22]:
import pandas as pd
import os

def load_all_stock_data(folder="./stock_data"):
    all_data = []

    for file in os.listdir(folder):
        if file.endswith(".csv"):
            ticker = file.split("_")[0]
            df = pd.read_csv(os.path.join(folder, file), parse_dates=["Date"])
            df.set_index("Date", inplace=True)

            # Use Close price 
            df = df[['Close']]
            df.rename(columns={'Close': ticker}, inplace=True)

            all_data.append(df)

    # Merge all stocks into one DataFrame
    combined_df = pd.concat(all_data, axis=1)

    # Drop rows with missing values
    combined_df = combined_df.dropna()

    return combined_df

# Load data
data =load_all_stock_data()
print(data.head(10))

# Compute returns
expected_return = data.pct_change().dropna()

# Covariance matrix
cov_matrix = expected_return.cov()

# Mean return & risk
mean_returns = expected_return.mean()
mean_risk = expected_return.std()

#score
score = mean_returns / mean_risk

metrics = pd.DataFrame({
            'Return': mean_returns,
            'Risk': mean_risk,
            'Score': score
        }).dropna()

# Sort by best stock
metrics = metrics.sort_values(by='Score', ascending=False)


                  AAPL        AMZN       GOOGL          GS         JPM  \
Date                                                                     
2024-01-02  183.731308  149.929993  137.037384  369.588898  163.843597   
2024-01-03  182.355621  148.470001  137.781250  363.392609  163.129471   
2024-01-04  180.039642  144.570007  135.271988  364.496674  164.212021   
2024-01-05  179.317154  145.240005  134.617386  367.818542  165.035934   
2024-01-08  183.652130  149.100006  137.701889  370.121857  164.796432   
2024-01-09  183.236465  151.369995  139.794601  365.248596  163.493530   
2024-01-10  184.275650  153.729996  141.113693  363.554382  163.838394   
2024-01-11  183.681793  155.179993  140.915344  361.450867  163.148636   
2024-01-12  184.008438  154.619995  141.480652  359.547272  161.951126   
2024-01-16  181.741989  153.160004  141.321976  362.117218  160.935654   

                  META        MSFT       NVDA        TSLA        WMT  
Date                                    

In [21]:
def greedy_portfolio_optimization(
    metrics,
    cov_matrix,
    budget=100,
    max_allocation=30,
    min_stocks=4,
    max_stocks=8,
    min_score_threshold=0.0
):
    """
    Greedy portfolio optimizer.
    
    Parameters
    ----------
    metrics             : DataFrame with columns Return, Risk, Score
    cov_matrix          : DataFrame covariance matrix of returns
    budget              : total weight budget (default 100%)
    max_allocation      : max weight per stock (default 30%)
    min_stocks          : minimum number of stocks to include
    max_stocks          : maximum number of stocks to include
    min_score_threshold : exclude stocks whose Score less than this (default 0.0)
    
    Returns
    -------
    portfolio_df : DataFrame of selected stocks with weights
    stats        : dict with portfolio-level Return, Risk, Sharpe
    """
    
    # Filter candidates by minimum score threshold
    candidates = metrics[metrics['Score'] >= min_score_threshold].copy()

    # If no of candidates is less than min_stocks, relax the threshold
    if len(candidates) < min_stocks:
        print(f"Warning: Only {len(candidates)} stocks passed threshold. Need {min_stocks}.")
        # fall back to top N stocks regardless of threshold
        candidates = metrics.head(min_stocks).copy()
    
    # Select the top N candidates
    selected_indices = candidates.index[:max_stocks]
    portfolio_df = candidates.loc[selected_indices].copy()
    portfolio_df['Allocation'] = 0.0

    remaining_budget = budget
    
    # --- Greedy selection ---
    for ticker in portfolio_df.index:
        if remaining_budget <= 0:
            break
        alloc = min(max_allocation, remaining_budget)
        portfolio_df.at[ticker, 'Allocation'] = alloc
        remaining_budget -= alloc

    while remaining_budget > 0.001:
        eligible = portfolio_df[portfolio_df['Allocation'] < max_allocation]
        if eligible.empty:
            break
        
        extra = remaining_budget / len(eligible)
        for ticker in eligible.index:
            current = portfolio_df.at[ticker, 'Allocation']
            take = min(extra, max_allocation - current)
            portfolio_df.at[ticker, 'Allocation'] += take
            remaining_budget -= take

    # Normalize to Weights (0.0 - 1.0)
    portfolio_df['Weight'] = portfolio_df['Allocation'] / portfolio_df['Allocation'].sum()

    # Portfolio risk
    weights = portfolio_df['Weight'].values

    # Subset cov matrix to only selected tickers
    sub_cov = cov_matrix.loc[selected_indices, selected_indices].values
    port_variance = np.dot(weights.T, np.dot(sub_cov, weights))
    port_risk = np.sqrt(port_variance)
    
    port_return = (portfolio_df['Weight'] * portfolio_df['Return']).sum()
    sharpe = port_return / port_risk if port_risk > 0 else 0.0

    stats = {
        'Expected Daily Return': port_return,
        'Portfolio Risk': port_risk,
        'Sharpe Ratio': sharpe,
        'No of Stocks': len(portfolio_df),
        'Total Weight': portfolio_df['Weight'].sum()
    }
    
    return portfolio_df, stats
    


def print_portfolio_report(portfolio_df, stats):
    print("=" * 55)
    print("  GREEDY PORTFOLIO OPTIMIZATION — RESULTS")
    print("=" * 55)
    
    print(f"\n{'Ticker':<10} {'Return':>10} {'Risk':>10} {'Score':>10} {'Weight':>10}")
    print("-" * 55)
    for ticker, row in portfolio_df.iterrows():
        print(f"{ticker:<10} {row['Return']:>9.4%} {row['Risk']:>9.4%} "
              f"{row['Score']:>10.4f} {row['Weight']:>9.1%}")
    
    print("\n--- Portfolio Summary ---")
    for k, v in stats.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")
    print("=" * 55)


# Perform optimization
portfolio_df, stats = greedy_portfolio_optimization(
    metrics,
    cov_matrix,
    budget=100,
    max_allocation=30,   
    min_stocks=4,        
    max_stocks=8,        
    min_score_threshold=0.0
)

print_portfolio_report(portfolio_df, stats)

  GREEDY PORTFOLIO OPTIMIZATION — RESULTS

Ticker         Return       Risk      Score     Weight
-------------------------------------------------------
WMT          0.1616%   1.3676%     0.1181     30.0%
GS           0.1881%   1.8010%     0.1044     30.0%
NVDA         0.3222%   3.2181%     0.1001     30.0%
GOOGL        0.1829%   1.9062%     0.0960     10.0%
JPM          0.1457%   1.5225%     0.0957      0.0%
META         0.1572%   2.3458%     0.0670      0.0%
AMZN         0.1056%   1.9781%     0.0534      0.0%
AAPL         0.0933%   1.7571%     0.0531      0.0%

--- Portfolio Summary ---
  Expected Daily Return: 0.0022
  Portfolio Risk: 0.0150
  Sharpe Ratio: 0.1466
  No of Stocks: 8
  Total Weight: 1.0000
